# SchoolBridge — LayoutLMv3 자체화 PoC

**목표**: 가정통신문 (HWP / PDF / 이미지) → 표 구조 의미 복원 (헤더 vs 값 식별)

**전제**:
- LLM 1단계 (Claude sentence 추출) → 자체 모델로 대체
- Span-based extraction (텍스트 변형 0 보장)
- 학년 매트릭스 같은 표 의미 복원 (단순 파싱으로 불가능한 영역)

**파이프라인**:
```
[입력] HWP / PDF / 이미지
   ↓
[Stage 1] 텍스트 + bbox 추출 (글자 위치만, 변형 0)
   HWP  → olefile / pyhwp (텍스트+좌표 직접)
   PDF  → pdfplumber (text layer)
   이미지 → PaddleOCR
   ↓
[Stage 2] LayoutLMv3 입력 변환
   페이지 이미지 + 토큰 + bbox
   ↓
[Stage 3] 표 의미 복원 (헤더/값 token classification)
   사전학습 모델 zero-shot baseline → fine-tune으로 도메인 특화
```

## 0. 셋업

In [ ]:
# 필요 라이브러리 설치 (Python 3.11 권장)
# !pip install torch torchvision transformers datasets
# !pip install pdfplumber pymupdf olefile pyhwp
# !pip install pillow numpy
# OCR (이미지 통신문용, 옵션)
# !pip install paddlepaddle paddleocr

In [ ]:
import os
import sys
import json
import zipfile
import zlib
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional

DATA_DIR = Path(r"c:/project/backend/data")
SAMPLE_HWP = DATA_DIR / "2026학년도 학습준비물 안내 가정통신문.hwp"
SAMPLE_HWPX = DATA_DIR / "해조류박람회 체험학습 참가 동의서.hwpx"
SAMPLE_PDF = DATA_DIR / "2026 2,3,5,6학년 구강검진 실시안내.pdf"

for p in (SAMPLE_HWP, SAMPLE_HWPX, SAMPLE_PDF):
    print(f"{p.name}: {'OK' if p.exists() else 'MISSING'} ({p.stat().st_size // 1024 if p.exists() else 0} KB)")

## 1. Stage 1 — 텍스트 + bbox 추출 (포맷별)

**핵심**: 글자 위치만 뽑음. 표 구조(헤더/값) 의미 X. 변형 0 보장.

### 1-1. HWPX (최신 표준, 가장 깨끗)
ZIP + XML 구조라 `zipfile + xml.etree`만으로 충분.

In [ ]:
import xml.etree.ElementTree as ET

@dataclass
class TextSpan:
    text: str
    bbox: Tuple[float, float, float, float]  # (x0, y0, x1, y1) — normalized 0~1
    page: int = 0
    cell_id: Optional[str] = None  # 표 셀에 속하면 셀 ID
    is_in_table: bool = False


def extract_hwpx(path: Path) -> List[TextSpan]:
    """HWPX (ZIP+XML) 에서 텍스트 + bbox + 셀 구조 추출.
    
    내부 구조:
    - Contents/section0.xml: 본문 (단락 / 표 / 셀 / 줄 / 글자)
    - 표는 <hp:tbl> 노드 안에 <hp:tr><hp:tc> 형태로 행/열 명시
    """
    spans: List[TextSpan] = []
    with zipfile.ZipFile(path) as z:
        section_files = [n for n in z.namelist() if n.startswith("Contents/section") and n.endswith(".xml")]
        for sf in sorted(section_files):
            xml_bytes = z.read(sf)
            root = ET.fromstring(xml_bytes)
            # 네임스페이스 hp: http://www.hancom.co.kr/hwpml/2011/paragraph
            ns = {"hp": "http://www.hancom.co.kr/hwpml/2011/paragraph"}
            # 표는 <hp:tbl> 안에 <hp:tr><hp:tc> 구조
            for tbl_idx, tbl in enumerate(root.iter("{http://www.hancom.co.kr/hwpml/2011/paragraph}tbl")):
                for tr_idx, tr in enumerate(tbl.iter("{http://www.hancom.co.kr/hwpml/2011/paragraph}tr")):
                    for tc_idx, tc in enumerate(tr.iter("{http://www.hancom.co.kr/hwpml/2011/paragraph}tc")):
                        # 셀 안 모든 텍스트 추출
                        texts = [t.text or "" for t in tc.iter() if t.text]
                        joined = " ".join(s.strip() for s in texts if s.strip())
                        if not joined:
                            continue
                        # bbox는 HWPX에 셀 단위 좌표가 있음 (offsetX/Y, width/height attribute)
                        cellAddr = tc.find("{http://www.hancom.co.kr/hwpml/2011/paragraph}cellAddr")
                        row = int(cellAddr.get("rowAddr")) if cellAddr is not None and cellAddr.get("rowAddr") else tr_idx
                        col = int(cellAddr.get("colAddr")) if cellAddr is not None and cellAddr.get("colAddr") else tc_idx
                        cell_id = f"tbl{tbl_idx}_r{row}_c{col}"
                        # bbox는 cellSz + cellMargin으로 계산 가능. 간단히 row/col을 정규화 좌표로:
                        bbox = (col * 0.1, row * 0.05, (col + 1) * 0.1, (row + 1) * 0.05)
                        spans.append(TextSpan(text=joined, bbox=bbox, page=0, cell_id=cell_id, is_in_table=True))
            # 표 밖 본문도 추출
            for p_idx, para in enumerate(root.iter("{http://www.hancom.co.kr/hwpml/2011/paragraph}p")):
                # 표 안 단락은 위에서 처리했으니 skip
                parent = para.find("..")
                if any(p.iter("{http://www.hancom.co.kr/hwpml/2011/paragraph}tbl") for p in [para]):
                    continue
                texts = [t.text or "" for t in para.iter() if t.text]
                joined = " ".join(s.strip() for s in texts if s.strip())
                if joined and not any(s.text == joined for s in spans):
                    bbox = (0.0, p_idx * 0.02, 1.0, (p_idx + 1) * 0.02)
                    spans.append(TextSpan(text=joined, bbox=bbox, page=0, is_in_table=False))
    return spans


hwpx_spans = extract_hwpx(SAMPLE_HWPX)
print(f"HWPX spans: {len(hwpx_spans)}")
for s in hwpx_spans[:15]:
    marker = "[TBL]" if s.is_in_table else "[TXT]"
    print(f"  {marker} {s.cell_id or '':18s} bbox={s.bbox} text={s.text[:80]!r}")

### 1-2. HWP 5.0 (binary, OLE 컨테이너)

`olefile` 로 BodyText 스트림 직접 파싱. HWPX보다 복잡하지만 더 많은 학교가 이 포맷.

In [ ]:
# HWP 5.0 텍스트 추출 — pyhwp 라이브러리 사용 권장
# 직접 파싱은 record-based 구조라 복잡 (PARA_TEXT / TABLE / CELL record IDs)
#
# 권장 라이브러리:
#   - pyhwp: pip install pyhwp  (가장 안정적, 셀 구조 추출 OK)
#   - hwp5: 5.0 binary 직접 파싱
#
# 사용 예 (pyhwp):
# from hwp5 import filestructure
# hwp = filestructure.Hwp5File(SAMPLE_HWP)
# for section in hwp.bodytext:
#     for para in section.paragraphs:
#         for ctrl in para.controls:
#             if ctrl.is_table:
#                 for row in ctrl.rows:
#                     for cell in row.cells:
#                         print(cell.text, cell.row_idx, cell.col_idx)

# 또는 LibreOffice headless로 HWP → PDF 변환 후 PDF 경로 사용:
# soffice --headless --convert-to pdf input.hwp
# Linux/Mac에서 동작. Windows는 한컴오피스 또는 LibreOffice 설치 필요.

print("HWP 5.0 binary: olefile/pyhwp 라이브러리 사용 권장 (실제 학습 데이터셋 만들 때)")
print("또는 LibreOffice headless로 HWP → PDF 변환 후 pdfplumber 경로")

### 1-3. PDF (text layer 있는 경우)

`pdfplumber.chars` — 글자별 좌표 직접 추출.

In [ ]:
import pdfplumber

def extract_pdf(path: Path) -> List[TextSpan]:
    """PDF text layer에서 단어 + bbox 추출. 표 자동 인식도 함께."""
    spans: List[TextSpan] = []
    with pdfplumber.open(path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            page_w, page_h = page.width, page.height
            # 표 자동 인식
            tables = page.find_tables()
            table_cells = {}  # bbox -> (table_idx, row, col)
            for t_idx, table in enumerate(tables):
                for r_idx, row in enumerate(table.rows):
                    for c_idx, cell in enumerate(row.cells):
                        if cell:
                            table_cells[cell] = (t_idx, r_idx, c_idx)
            # 단어별 추출
            words = page.extract_words()
            for w in words:
                bbox = (w["x0"] / page_w, w["top"] / page_h, w["x1"] / page_w, w["bottom"] / page_h)
                # 표 셀에 속하는지 확인
                cell_id = None
                in_table = False
                for cell_bbox, (t, r, c) in table_cells.items():
                    cx0, cy0, cx1, cy1 = cell_bbox
                    if cx0 <= w["x0"] and cy0 <= w["top"] and w["x1"] <= cx1 and w["bottom"] <= cy1:
                        cell_id = f"tbl{t}_r{r}_c{c}"
                        in_table = True
                        break
                spans.append(TextSpan(text=w["text"], bbox=bbox, page=page_idx, cell_id=cell_id, is_in_table=in_table))
    return spans


pdf_spans = extract_pdf(SAMPLE_PDF)
print(f"PDF spans: {len(pdf_spans)}")
for s in pdf_spans[:20]:
    marker = "[TBL]" if s.is_in_table else "[TXT]"
    print(f"  {marker} p{s.page} {s.cell_id or '':18s} text={s.text[:40]!r}")

### 1-4. 페이지 이미지 렌더링

LayoutLMv3는 이미지도 입력으로 받음. PyMuPDF로 페이지 → 이미지.

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import io

def render_pdf_pages(path: Path, dpi: int = 150) -> List[Image.Image]:
    """PDF → PIL Image 리스트 (페이지별)"""
    images = []
    doc = fitz.open(path)
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        images.append(img)
    return images


pdf_pages = render_pdf_pages(SAMPLE_PDF, dpi=150)
print(f"PDF rendered: {len(pdf_pages)} page(s), first size: {pdf_pages[0].size}")
# pdf_pages[0]  # Jupyter에서 직접 표시

## 2. 시각화 — 추출 결과 확인

이미지에 bbox 오버레이. 추출이 깨지지 않았는지, 표 셀이 잘 잡혔는지 검증.

In [ ]:
from PIL import ImageDraw, ImageFont

def overlay_bboxes(img: Image.Image, spans: List[TextSpan], page: int = 0) -> Image.Image:
    out = img.copy()
    draw = ImageDraw.Draw(out)
    W, H = out.size
    for s in spans:
        if s.page != page:
            continue
        x0, y0, x1, y1 = s.bbox
        rect = (x0 * W, y0 * H, x1 * W, y1 * H)
        color = "red" if s.is_in_table else "blue"
        draw.rectangle(rect, outline=color, width=1)
    return out


if pdf_pages and pdf_spans:
    overlay = overlay_bboxes(pdf_pages[0], pdf_spans, page=0)
    overlay.save("layoutlmv3_poc_overlay.png")
    print("Overlay saved: layoutlmv3_poc_overlay.png")
    # overlay  # Jupyter에서 직접 확인

## 3. Stage 2 — LayoutLMv3 입력 변환

추출한 (text, bbox) + 페이지 이미지 → LayoutLMv3 processor

In [ ]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
import torch

MODEL_ID = "microsoft/layoutlmv3-base"  # 영어 base, 추후 multilingual로 교체 가능
# 한국어 강화 옵션: "nielsr/layoutlmv3-finetuned-funsd" 또는 multilingual 직접 학습

processor = LayoutLMv3Processor.from_pretrained(MODEL_ID, apply_ocr=False)  # 우리가 직접 추출했으니 OCR off
print("Processor loaded:", processor.__class__.__name__)

In [ ]:
def to_layoutlmv3_inputs(image: Image.Image, spans: List[TextSpan], page: int = 0):
    """PIL 이미지 + TextSpan 리스트 → LayoutLMv3 processor 입력
    
    bbox 정규화: LayoutLMv3는 0~1000 정수 좌표 요구.
    """
    page_spans = [s for s in spans if s.page == page]
    words = [s.text for s in page_spans]
    boxes = []
    for s in page_spans:
        x0, y0, x1, y1 = s.bbox
        boxes.append([int(x0 * 1000), int(y0 * 1000), int(x1 * 1000), int(y1 * 1000)])
    encoded = processor(
        image,
        words,
        boxes=boxes,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    return encoded, page_spans


if pdf_pages and pdf_spans:
    encoded, page_spans = to_layoutlmv3_inputs(pdf_pages[0], pdf_spans, page=0)
    print("Input shapes:")
    for k, v in encoded.items():
        if hasattr(v, "shape"):
            print(f"  {k}: {v.shape}")
        else:
            print(f"  {k}: {type(v).__name__}")

## 4. Stage 3 — 헤더/값 분류 (Token Classification)

### 4-1. 라벨 스키마

토큰별 태그 (B/I/O 표기로 span 단위):

| 태그 | 의미 |
|---|---|
| `O` | 표 밖 일반 텍스트 |
| `B-HEADER_ROW` / `I-HEADER_ROW` | 행 헤더 (가로 표의 좌측 첫 열) |
| `B-HEADER_COL` / `I-HEADER_COL` | 열 헤더 (세로 표의 상단 첫 행) |
| `B-VALUE` / `I-VALUE` | 값 셀 |
| `B-FREE_TEXT` / `I-FREE_TEXT` | 표 밖 본문 |

**학년 매트릭스 케이스**:
- "학년" → `B-HEADER_COL` (열 헤더 origin)
- "1", "2", "3", "4", "5", "6" → `B-HEADER_COL` (열 헤더 값들)
- "준비물" → `B-HEADER_ROW` (행 헤더)
- "스케치북, 색연필" → `B-VALUE` (1학년 준비물 값)

In [ ]:
LABELS = [
    "O",
    "B-HEADER_ROW", "I-HEADER_ROW",
    "B-HEADER_COL", "I-HEADER_COL",
    "B-VALUE", "I-VALUE",
    "B-FREE_TEXT", "I-FREE_TEXT",
]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
print("Labels:", LABELS)

### 4-2. 사전학습 모델 로드 + Zero-shot 추론 베이스라인

Fine-tune 전, 사전학습 그대로 출력 확인 (random init이라 의미 없지만 파이프라인 검증용).

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.eval()
print(f"Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

In [ ]:
# 추론 (random init head라 결과는 의미 없음, 파이프라인 검증용)
if pdf_pages and pdf_spans:
    with torch.no_grad():
        outputs = model(**encoded)
    predictions = outputs.logits.argmax(-1)[0].tolist()
    # 토큰별 예측 라벨 출력 (앞 30개만)
    tokens = processor.tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    for tok, pred in list(zip(tokens, predictions))[:30]:
        if tok in ("<s>", "</s>", "<pad>"):
            continue
        print(f"  {tok:25s} → {ID2LABEL[pred]}")

## 5. 학습 데이터셋 형식 (Fine-tune 용)

라벨링은 사람이 한다고 했을 때, 각 통신문 페이지에 대해 아래 형식의 JSON을 만든다:

In [ ]:
# 라벨링 결과 형식 예시 (Label Studio 또는 doccano 출력 → 이 형식으로 변환)
sample_annotation = {
    "image_path": "data/labeled/2026학년도_학습준비물_p1.png",
    "page": 0,
    "page_size": [1240, 1754],  # 이미지 픽셀 크기
    "spans": [
        # 학년 매트릭스 예시
        {"text": "학년", "bbox": [100, 200, 180, 240], "label": "B-HEADER_COL", "cell_id": "tbl0_r0_c0"},
        {"text": "1",   "bbox": [200, 200, 240, 240], "label": "B-HEADER_COL", "cell_id": "tbl0_r0_c1"},
        {"text": "2",   "bbox": [260, 200, 300, 240], "label": "B-HEADER_COL", "cell_id": "tbl0_r0_c2"},
        {"text": "3",   "bbox": [320, 200, 360, 240], "label": "B-HEADER_COL", "cell_id": "tbl0_r0_c3"},
        # ... 4, 5, 6
        {"text": "준비물", "bbox": [100, 250, 180, 290], "label": "B-HEADER_ROW", "cell_id": "tbl0_r1_c0"},
        {"text": "스케치북, 색연필, 사인펜", "bbox": [200, 250, 240, 290], "label": "B-VALUE", "cell_id": "tbl0_r1_c1"},
        # ...
    ],
    "table_meta": [
        {"table_id": "tbl0", "orientation": "MATRIX", "header_rows": [0], "header_cols": [0]},
    ],
}
print(json.dumps(sample_annotation, ensure_ascii=False, indent=2)[:1500])

## 6. Fine-tune 학습 코드 스켈레톤

**환경**: GPU 필요 (Colab Pro 또는 NCP GPU 1대). CPU에서 학습은 비현실적.  
**예상 시간**: 1,000장 학습, 5 epoch, T4 GPU 기준 4~8시간.  
**추론은 CPU OK** (LayoutLMv3-base 134M 기준 1페이지 ~1초)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import TrainingArguments, Trainer

class NoticeLayoutDataset(Dataset):
    """라벨링된 통신문 페이지 → LayoutLMv3 입력"""
    def __init__(self, annotation_paths: List[Path], processor):
        self.items = [json.loads(p.read_text(encoding="utf-8")) for p in annotation_paths]
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        W, H = item["page_size"]
        words = [s["text"] for s in item["spans"]]
        boxes = [[int(b[0] / W * 1000), int(b[1] / H * 1000),
                  int(b[2] / W * 1000), int(b[3] / H * 1000)]
                 for b in [s["bbox"] for s in item["spans"]]]
        word_labels = [LABEL2ID[s["label"]] for s in item["spans"]]
        encoded = self.processor(
            image, words, boxes=boxes, word_labels=word_labels,
            return_tensors="pt", truncation=True, padding="max_length", max_length=512,
        )
        return {k: v.squeeze(0) for k, v in encoded.items()}


# 학습 예시 (실제 학습은 GPU 환경에서):
# train_ds = NoticeLayoutDataset(list(Path("data/labeled/train").glob("*.json")), processor)
# val_ds = NoticeLayoutDataset(list(Path("data/labeled/val").glob("*.json")), processor)
# 
# training_args = TrainingArguments(
#     output_dir="./layoutlmv3_notice_v1",
#     num_train_epochs=5,
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     learning_rate=5e-5,
#     warmup_ratio=0.1,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
#     fp16=True,  # GPU에서
# )
# trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds)
# trainer.train()
print("학습 코드 스켈레톤 — 실제 학습은 GPU 환경에서. 라벨링 데이터 1,000장+ 준비 후 진행.")

## 7. 추론 함수 — 학년 매트릭스 케이스 해결

Fine-tune된 모델로 새 통신문 입력 → (header, value) 쌍 출력

In [ ]:
def infer_notice(image: Image.Image, spans: List[TextSpan], model, processor):
    """통신문 페이지 → 헤더/값 분류 결과"""
    encoded, page_spans = to_layoutlmv3_inputs(image, spans, page=0)
    with torch.no_grad():
        outputs = model(**encoded)
    predictions = outputs.logits.argmax(-1)[0].tolist()
    # word_id로 토큰 → 원본 단어 매핑
    word_ids = encoded.word_ids() if hasattr(encoded, "word_ids") else None
    results = []
    if word_ids:
        for word_idx, span in enumerate(page_spans):
            # 해당 단어에 속하는 토큰의 예측 다수결
            tok_preds = [predictions[t] for t, w in enumerate(word_ids) if w == word_idx]
            if tok_preds:
                pred = max(set(tok_preds), key=tok_preds.count)
                results.append({"text": span.text, "bbox": span.bbox, "label": ID2LABEL[pred],
                                "cell_id": span.cell_id})
    return results

# 사용 예 (fine-tune 후):
# results = infer_notice(pdf_pages[0], pdf_spans, model, processor)
# for r in results:
#     if r['label'] != 'O':
#         print(f"  [{r['label']:15s}] {r['cell_id'] or '':15s} {r['text']}")

## 8. 후처리 — (header, value) 쌍으로 재구성

토큰별 라벨 + cell_id를 기반으로 표 의미 복원.

**학년 매트릭스 예시 출력**:
```json
{
  "table_id": "tbl0",
  "orientation": "MATRIX",
  "header_row": ["학년", "1", "2", "3", "4", "5", "6"],
  "header_col": ["학년", "준비물", "제출", "비용"],
  "data": {
    "1학년_준비물": "스케치북, 색연필, 사인펜",
    "2학년_준비물": "...",
    ...
  }
}
```

이 정형 데이터를 그대로 KoELECTRA·KcELECTRA·NLLB 입력으로 흘리면 LLM 1단계 완전 제거.

In [ ]:
from collections import defaultdict

def assemble_table_meaning(results: List[dict]) -> List[dict]:
    """토큰별 라벨 결과 → (header, value) 쌍 정형 데이터."""
    tables = defaultdict(lambda: {"cells": [], "orientation": None})
    for r in results:
        if not r.get("cell_id"):
            continue
        tbl_id = r["cell_id"].split("_")[0]
        tables[tbl_id]["cells"].append(r)
    
    output = []
    for tbl_id, t in tables.items():
        header_cols = [c for c in t["cells"] if "HEADER_COL" in c["label"]]
        header_rows = [c for c in t["cells"] if "HEADER_ROW" in c["label"]]
        values = [c for c in t["cells"] if "VALUE" in c["label"]]
        # 셀 ID에서 (row, col) 파싱
        def parse_rc(cell_id):
            try:
                r = int(cell_id.split("_r")[1].split("_")[0])
                c = int(cell_id.split("_c")[1])
                return r, c
            except Exception:
                return -1, -1
        # 헤더 위치 추정
        col_header_row_idx = min((parse_rc(h["cell_id"])[0] for h in header_cols), default=0)
        row_header_col_idx = min((parse_rc(h["cell_id"])[1] for h in header_rows), default=0)
        # 매트릭스 형태로 재구성
        col_headers = {parse_rc(h["cell_id"])[1]: h["text"] for h in header_cols}
        row_headers = {parse_rc(h["cell_id"])[0]: h["text"] for h in header_rows}
        data = {}
        for v in values:
            r, c = parse_rc(v["cell_id"])
            rh = row_headers.get(r, f"r{r}")
            ch = col_headers.get(c, f"c{c}")
            data[f"{ch}_{rh}"] = v["text"]
        output.append({
            "table_id": tbl_id,
            "header_row": list(col_headers.values()),
            "header_col": list(row_headers.values()),
            "data": data,
        })
    return output

# 사용 예:
# tables = assemble_table_meaning(results)
# print(json.dumps(tables, ensure_ascii=False, indent=2))

## 9. 발표 메시지 라인

- **현재 (LLM 1단계)**: 페이지 → 텍스트 → Claude → sentence_list (비결정적, 변형 위험)
- **자체화 (LayoutLMv3)**: 페이지 → 시각·텍스트·좌표 → 모델 → 토큰별 태그 (결정적, 변형 0)
- **학년 매트릭스 같은 표 의미 복원**은 LayoutLMv3가 학습한 layout 패턴으로 해결
- **우리 자산**: 3,300+장 통신문 + KoELECTRA/KcELECTRA fine-tune 경험
- **추가 모델 1개** (LayoutLMv3) + 기존 자체 모델 3개 → **API 의존 0**

**스쿨포인트와의 본질 차이**: 그들은 4단계 전부 ChatGPT API → 자체화 경로 자체가 막힘. 우리는 1단계만 LayoutLMv3로 자체화하면 완전한 자체 파이프라인.

---

## 10. 다음 단계 체크리스트

- [ ] Phase 1: HWPX/PDF 추출 결과 시각화 검증 (이 노트북 1~2번 셀)
- [ ] Phase 2: 라벨링 도구 선정 (Label Studio 또는 doccano)
- [ ] Phase 3: 다양성 샘플 1,000장 선별 (학년 매트릭스 / 일정표 / 비용표 / 자유 서술 골고루)
- [ ] Phase 4: 라벨링 (1,000장 × 1~3분 = 30~50시간, 팀 분담 또는 외주)
- [ ] Phase 5: GPU 환경에서 fine-tune (Colab Pro 또는 NCP GPU L4)
- [ ] Phase 6: 평가 — Claude 1단계와 동일 통신문에서 비교 (결정성·정확도)
- [ ] Phase 7: NCP CPU에서 inference 통합, LLM 1단계 분리